In [1]:
from the_well.benchmark.metrics.spatial import VRMSE
import matplotlib.pyplot as plt
from einops import rearrange
import seaborn as sns
import pandas as pd
import numpy as np
import pickle
import torch
import math
import sys

rc = {
    "axes.axisbelow": True,
    "axes.linewidth": 1.0,
    "axes.labelsize": "x-small",
    "axes.titlesize": "x-small",
    "figure.dpi": 150,
    "figure.figsize": (3.2, 3.2),
    "font.size": 12.0,
    "legend.fontsize": "xx-small",
    "lines.linewidth": 1.0,
    "savefig.transparent": True,
    "xtick.labelsize": "xx-small",
    "xtick.major.width": 1.0,
    "ytick.labelsize": "xx-small",
    "ytick.major.width": 1.0,
    "font.family": ["serif"],
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsfonts}",
}

plt.rcParams.update(rc)

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.eval import build_stats_df, build_psd_df
from waveletflux.utils.plot import plot_psd, plot_psd_seaborn
#from waveletflux.utils.plot import plot_psd_seaborn
from waveletflux.utils.well_data import get_well_dataset

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

outdir = f'{PROJECT_ROOT}/figures'

# Model checklist 

In [ ]:
model_checklist = [
    'WaveletFlow_3scales_db2',
    ]

# Load Stats

In [11]:
import glob
import pandas as pd
from pathlib import Path

well_dataset_name = 'planetswe'

dfs = []

for model in model_checklist:
    pattern = f'{PROJECT_ROOT}/experiments/{well_dataset_name}/metrics/{model}*stats.csv'
    matched_files = glob.glob(pattern)

    if not matched_files:
        print(f"[WARN] No files found for model '{model}' with pattern: {pattern}")
        continue

    for fpath in sorted(matched_files):
        try:
            df = pd.read_csv(fpath)

            # Parse metadata from filename
            # e.g. WaveletFlow_3scales_db2_v1.0_planetswe_nsampling20_nens8_stats.csv
            stem = Path(fpath).stem  # strip .csv
            parts = stem.split('_')

            # Extract nsampling and nens if present
            nsampling = next((p.replace('nsampling', '') for p in parts if p.startswith('nsampling')), None)
            nens      = next((p.replace('nens',      '') for p in parts if p.startswith('nens')),      None)

            df['model']     = model
            df['source']    = stem
            df['dataset']   = well_dataset_name
            df['nsampling'] = int(nsampling) if nsampling else None
            df['nens']      = int(nens)      if nens      else None

            dfs.append(df)
            print(f"  [OK] {Path(fpath).name:70s}  →  {df.shape}")

        except Exception as e:
            print(f"  [ERR] {fpath}: {e}")

if dfs:
    metrics_df = pd.concat(dfs, ignore_index=True)
    print(f"\nMerged DataFrame: {metrics_df.shape}  |  columns: {list(metrics_df.columns)}")
    display(metrics_df.head())
else:
    metrics_df = pd.DataFrame()
    print("[WARN] No data loaded — metrics_df is empty.")

  [OK] WaveletFlow_3scales_db2_v1.0_planetswe_nsampling20_nens8_stats.csv      →  (1968, 28)

Merged DataFrame: (1968, 28)  |  columns: ['model', 'traj', 'field', 'step', 'time', 'm1', 'm2', 'spread', 'rmse', 'nrmse', 'vrmse', 'spread_skill', 'total_error', 'w1', 'emd', 'crps', 'rmse_p_low', 'rmse_c_low', 'rmse_p_mid', 'rmse_c_mid', 'rmse_p_high', 'rmse_c_high', 'rmse_p_vhigh', 'rmse_c_vhigh', 'source', 'dataset', 'nsampling', 'nens']


,model,traj,field,step,time,m1,m2,spread,rmse,nrmse,...,rmse_p_mid,rmse_c_mid,rmse_p_high,rmse_c_high,rmse_p_vhigh,rmse_c_vhigh,source,dataset,nsampling,nens
0,WaveletFlow_3scales_db2,0,0,0,0,-94.039871,91549.320312,4.236998,5.397947,0.017840,...,0.038117,0.001431,0.034924,0.004551,236150752.0,0.377286,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
1,WaveletFlow_3scales_db2,0,0,1,1,-94.448898,91921.203125,4.835152,6.306252,0.020800,...,0.017553,0.002313,0.037794,0.006929,172110704.0,0.370506,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
2,WaveletFlow_3scales_db2,0,0,2,2,-95.132828,93241.226562,5.409786,8.924844,0.029228,...,0.025892,0.003104,0.079463,0.010160,206039744.0,0.339372,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
3,WaveletFlow_3scales_db2,0,0,3,3,-95.949295,94414.156250,5.968484,11.405330,0.037118,...,0.041014,0.003656,0.118916,0.016633,274511200.0,0.358653,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
4,WaveletFlow_3scales_db2,0,0,4,4,-96.646896,94942.218750,7.552678,12.361610,0.040119,...,0.029508,0.003821,0.131635,0.026713,618627072.0,0.345251,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8


In [12]:
df

,model,traj,field,step,time,m1,m2,spread,rmse,nrmse,...,rmse_p_mid,rmse_c_mid,rmse_p_high,rmse_c_high,rmse_p_vhigh,rmse_c_vhigh,source,dataset,nsampling,nens
0,WaveletFlow_3scales_db2,0,0,0,0,-94.039871,91549.320312,4.236998,5.397947,0.017840,...,0.038117,0.001431,0.034924,0.004551,236150752.0,0.377286,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
1,WaveletFlow_3scales_db2,0,0,1,1,-94.448898,91921.203125,4.835152,6.306252,0.020800,...,0.017553,0.002313,0.037794,0.006929,172110704.0,0.370506,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
2,WaveletFlow_3scales_db2,0,0,2,2,-95.132828,93241.226562,5.409786,8.924844,0.029228,...,0.025892,0.003104,0.079463,0.010160,206039744.0,0.339372,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
3,WaveletFlow_3scales_db2,0,0,3,3,-95.949295,94414.156250,5.968484,11.405330,0.037118,...,0.041014,0.003656,0.118916,0.016633,274511200.0,0.358653,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
4,WaveletFlow_3scales_db2,0,0,4,4,-96.646896,94942.218750,7.552678,12.361610,0.040119,...,0.029508,0.003821,0.131635,0.026713,618627072.0,0.345251,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1963,WaveletFlow_3scales_db2,3,2,159,159,2.300091,163.720215,11.256313,13.286750,1.038407,...,2.018212,0.303674,1121.201294,0.747101,595118336.0,0.353394,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
1964,WaveletFlow_3scales_db2,3,2,160,160,2.299106,163.785248,11.247142,13.294212,1.038784,...,2.096494,0.302953,1238.769409,0.745979,619626944.0,0.338333,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
1965,WaveletFlow_3scales_db2,3,2,161,161,2.275620,164.065308,11.249686,13.312223,1.039303,...,2.374340,0.295856,1521.053223,0.738005,641531968.0,0.337519,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8
1966,WaveletFlow_3scales_db2,3,2,162,162,2.226405,164.738495,11.254313,13.328219,1.038424,...,2.719509,0.285188,1995.157959,0.725223,618307648.0,0.344396,WaveletFlow_3scales_db2_v1.0_planetswe_nsampli...,planetswe,20,8


In [13]:
aggregate_mode = "mean"

def horizon(t):
    if t < 20:
        return "01:33"
    elif t < 60:
        return "34:65"
    elif t < 100:
        return "66:97"
    else:
        return "101:180"
    
def aggregate(x):
    if "mean" in aggregate_mode:
        return np.mean(x)
    elif "median" in aggregate_mode:
        return np.median(x)

#def model_type(model):
#    if model in generative_models:
#        return "generative"
#    return "deterministic"